In [28]:
from si4dnn import InferenceModel
import torch
import numpy as np
import time

In [ ]:
model = torch.nn.Sequential(
        torch.nn.Linear(10, 500),
        torch.nn.LeakyReLU(),
        torch.nn.Linear(500, 500),
        torch.nn.ReLU(),
        torch.nn.Linear(500, 500),
        torch.nn.ReLU(),
        torch.nn.Linear(500, 500),
        torch.nn.ReLU(),
        torch.nn.Linear(500, 500),
        torch.nn.ReLU(),
        torch.nn.Linear(500, 500),
        torch.nn.ReLU(),
        torch.nn.Linear(500, 2)
    ) # Can be modified freely

In [31]:
# Training process
n_train = 1000
p = 10

# Generate synthetic training data
X_train = torch.randn(n_train, p)
y_train = torch.randn(n_train, 2)  # Target with 2 outputs to match model output

# Setup optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# Training loop
n_epochs = 100
batch_size = 64

model.train()
for epoch in range(n_epochs):
    # Shuffle data
    perm = torch.randperm(n_train)
    X_train = X_train[perm]
    y_train = y_train[perm]
    
    X_train = X_train.cuda()
    y_train = y_train.cuda()
    
    epoch_loss = 0.0
    n_batches = 0
    
    for i in range(0, n_train, batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {epoch_loss/n_batches:.4f}')

model.eval()
print('Training completed!')

Epoch [20/100], Loss: 0.1116
Epoch [40/100], Loss: 0.0030
Epoch [60/100], Loss: 0.0049
Epoch [80/100], Loss: 0.0082
Epoch [100/100], Loss: 0.0044
Training completed!


In [32]:
n = 150
p = 10


inference_model = InferenceModel(model, device='cuda') # Can be "cuda" also

# X = a + bz
a = np.random.randn(n, p).astype(np.float32)
b = np.random.randn(n, p).astype(np.float32)
z = np.random.randn(1).astype(np.float32)
X = a + b * z

d:\Workspace\Research\StatiscalLearning\si4dnn\si4dnn\CUDA\util.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  w = torch.tensor(w.T, device="cuda") if w is not None else None
d:\Workspace\Research\StatiscalLearning\si4dnn\si4dnn\CUDA\util.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  b = torch.tensor(b, device="cuda") if b is not None else None


In [34]:
# Find interval for the forward process and X_hat = model(X) = a_hat + b_hat * z 
X_hat = model(torch.from_numpy(X).cuda()).detach().cpu().numpy()
start = time.time()
a_hat, b_hat, model_itv, = inference_model.forward(a, b, z)
print("Time for forward process:", time.time() - start)
print(model_itv)

Time for forward process: 0.018996715545654297
[-1.307772159576416, -1.3077328205108643]
